In [1]:
import torch
import torch.nn as nn
from torch.nn.functional import cross_entropy
from torch.utils.data import DataLoader, Dataset

from gpt_arcitecture import GPTModel, GPTDataset

import tiktoken
import pandas as pd
from tqdm import tqdm

import warnings
warnings.filterwarnings("ignore")

if torch.cuda.is_available():
    torch.set_default_device("cuda")
device = torch.get_default_device()
generator = torch.Generator(device = device)

tokenizer = tiktoken.encoding_for_model("gpt2")

CONFIG = {
    "n_vocab": 50257,    # Vocabulary size
    "n_ctx": 256, # Context length
    "n_embd": 768,         # Embedding dimension
    "n_head": 12,          # Number of attention heads
    "n_layer": 12,         # Number of layers
    "drop_rate": 0.1,       # Dropout rate
    "qkv_bias": False       # Query-Key-Value bias
}

model = GPTModel(CONFIG)

train = False

2025-11-05 09:31:21.845401: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-05 09:31:22.280455: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-11-05 09:31:24.669894: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-05 09:31:24.672160: I external/local_xla/xla/tsl/cuda/cudart

In [2]:
data_df = pd.read_csv("./archive/job_dataset.csv", encoding="utf-8")
data_df = data_df.sample(frac=1).reset_index(drop=True)


data_series = data_df["Title"]+"-" +data_df["ExperienceLevel"] + " : " +  data_df["Responsibilities"]

train_text = ""
val_text = ""

train_ratio = 0.98
split_idx = int(len(data_series)*train_ratio)
print(split_idx)

for row in data_series[:split_idx]:
    train_text += str(row) + "<|endoftext|>"

for row in data_series[split_idx:]:
    val_text += str(row) + "<|endoftext|>"

len(train_text), len(val_text)

1046


(299906, 6194)

In [3]:
batch_size = 5

train_data = GPTDataset(train_text, tokenizer, CONFIG["n_ctx"], CONFIG["n_ctx"])
val_data = GPTDataset(val_text, tokenizer, CONFIG["n_ctx"], CONFIG["n_ctx"])
train_loader = DataLoader(train_data, batch_size, shuffle = True, drop_last = False, generator= generator)
val_loader = DataLoader(val_data, batch_size, shuffle = True, drop_last = False, generator= generator)

for x,y in train_loader:
    print(x.shape, y.shape)
    break
print(len(train_loader))
print(len(val_loader))

torch.Size([5, 256]) torch.Size([5, 256])
42
1


In [4]:
def calculate_loss_batch(input_batch, target_batch, model):
    pred_batch = model(input_batch)
    loss = cross_entropy(pred_batch.flatten(0,1), target_batch.flatten())

    return loss

def calculate_loss_loader(data_loader, model, num_batches=None):
    total_loss = 0

    for i, (input_batch, target_batch) in enumerate(data_loader):
        loss = calculate_loss_batch(input_batch, target_batch, model)
        total_loss += loss.item()

    return total_loss/len(data_loader)


def get_pred(model, inputs, output_tokens = 1, sample = False):

    for i in range(output_tokens):
        with torch.no_grad():
            output = model(inputs)

        last_row = output[:,-1,:]
        probs = torch.softmax(last_row, dim=-1)

        if sample:
            output = torch.multinomial(probs,num_samples = 1)
        else:
            output = probs.argmax(dim=-1,keepdim=True)[0][0]
        inputs.append(output.item())

    return inputs


with torch.no_grad():
    val_loss = calculate_loss_loader(val_loader, model)
    train_loss = calculate_loss_loader(train_loader, model)


print(train_loss, val_loss)

torch.cuda.empty_cache()

10.99204885391962 10.988275527954102


In [5]:
# training loop
history = []
iteration = 0
optimizer = torch.optim.AdamW(model.parameters(), lr=4e-4, weight_decay=0.1)
sample_input = "Python Developer-Fresher :"

In [6]:
epochs = 50
if not train:
    epochs = 0
for epoch in tqdm(range(epochs)):
    iteration += 1

    for input_batch, output_batch in train_loader:
        optimizer.zero_grad()

        loss = calculate_loss_batch(input_batch, output_batch, model)
        loss.backward()

        optimizer.step()

    with torch.no_grad():
        perplexity = torch.exp(loss).item()
        val_loss = calculate_loss_loader(val_loader, model)
        train_loss = calculate_loss_loader(train_loader, model)
        sample_output = tokenizer.decode(get_pred(model, tokenizer.encode(sample_input), 50, False))
    
    history.append({
        "Epoch" : iteration,
        "Training Loss": train_loss,
        "Validation Loss": val_loss,
        "Perplexity": perplexity,
        "Sample": sample_output
    })

    print(f"Epoch : {epoch+1} : Train Loss = {train_loss:.4f}, Validation Loss = {val_loss:.4f}, Perplexity = {perplexity:.2f}")
    print(f"Output : {sample_output}\n")

    torch.save({"model_state_dict" : model.state_dict(), "optim_state_dict":optimizer.state_dict()}, "./archive/job_modelandoptim.pth")
    history_df = pd.DataFrame(history)
    history_df.to_csv("./archive/job_history.csv", index=False)

0it [00:00, ?it/s]


In [7]:
model.load_state_dict(torch.load("./archive/job_modelandoptim.pth")["model_state_dict"])
history_df = pd.read_csv("./archive/job_history.csv")

In [8]:
import plotly.express as px

fig = px.line(history_df, "Epoch", ["Training Loss", "Validation Loss"])
fig.show()
fig = px.line(history_df, "Epoch", "Perplexity")
fig.show()

In [9]:
sample_output = tokenizer.decode(get_pred(model, tokenizer.encode("AI Engineer-Experienced :"), 250, True))
for sample in sample_output.split("<|endoftext|>"):
    print(sample)

AI Engineer-Experienced : Lead system design initiatives; Manage enterprise-scale infrastructure; Ensure compliance adherence; Optimize automation projects; Mentor junior engineers
IoT Engineer-Experienced : Lead architecture and development of IoT solutions; Optimize embedded firmware and device software; Integrate complex sensors and actuators; Manage cloud and edge computing platforms; Develop data pipelines and analytics workflows; Implement secure communication protocols; Mentor team members and guide project delivery
Product Designer-Fresher : Create low-fidelity prototypes for new product features; Support usability testing sessions; Assist in building information architecture diagrams; Collaborate with developers for front-end consistency; Document design decisions and iterations
Market Research Analyst-Fresher : Support market research projects and data collection; Assist in designing surveys and research instruments; Analyze basic data trends; Prepare reports and presentation